# ASM-Shadhin-AI: IEEE Manuscript Statistical Analysis
## N=30 Benchmark: Welch's t-test, Cohen's d, Shapiro-Wilk, Wilson CI
### Reproducible analysis notebook for peer reviewers

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import json, os, warnings
warnings.filterwarnings('ignore')

WS = '/Volumes/BSc Works/AI digital automated system for security monitoring'
csv_path = os.path.join(WS, 'testbed/statistical_30_runs_evaluation.csv')
df = pd.read_csv(csv_path)
print(f'Loaded {len(df)} rows from N=30 benchmark CSV')
print(df['framework'].value_counts())
df.head(8)

## 1. Descriptive Statistics (mean ± std, 95% CI)

In [ ]:
from scipy.stats import sem, t as t_dist

frameworks = ['iptables', 'Suricata Inline', 'DPDK', 'ASM-Shadhin-AI (eBPF/XDP)']
metrics = ['throughput_mpps', 'latency_mean_us', 'jitter_us', 'cpu_util_pct']

print(f'{'Framework':<35} {'Metric':<22} {'Mean':>8} {'Std':>8} {'95% CI Lo':>10} {'95% CI Hi':>10}')
print('-'*100)
for fw in frameworks:
    sub = df[df['framework']==fw]
    for m in metrics:
        vals = sub[m].values
        mn, sd = np.mean(vals), np.std(vals, ddof=1)
        ci = stats.t.interval(0.95, df=len(vals)-1, loc=mn, scale=sem(vals))
        print(f'{fw:<35} {m:<22} {mn:>8.4f} {sd:>8.4f} {ci[0]:>10.4f} {ci[1]:>10.4f}')
    print()

## 2. Shapiro-Wilk Normality Test

In [ ]:
print('Shapiro-Wilk Normality Test (H0: data is normally distributed)')
print(f'{'Framework':<35} {'Metric':<22} {'W-stat':>8} {'p-value':>10} {'Normal?':>10}')
print('-'*90)
for fw in frameworks:
    sub = df[df['framework']==fw]
    for m in ['throughput_mpps', 'latency_mean_us']:
        vals = sub[m].values
        W, p = stats.shapiro(vals)
        normal = 'YES' if p > 0.05 else 'NO'
        print(f'{fw:<35} {m:<22} {W:>8.4f} {p:>10.6f} {normal:>10}')

## 3. Welch's Two-Sample Independent t-Test + Cohen's d

In [ ]:
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na-1)*np.std(a,ddof=1)**2 + (nb-1)*np.std(b,ddof=1)**2) / (na+nb-2))
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std > 0 else 0

asm = df[df['framework']=='ASM-Shadhin-AI (eBPF/XDP)']
ipt = df[df['framework']=='iptables']
sur = df[df['framework']=='Suricata Inline']

comparisons = [
    ('Throughput: ASM vs Netfilter', asm['throughput_mpps'], ipt['throughput_mpps']),
    ('Throughput: ASM vs Suricata',  asm['throughput_mpps'], sur['throughput_mpps']),
    ('Latency:    ASM vs Netfilter', asm['latency_mean_us'], ipt['latency_mean_us']),
    ('Latency:    ASM vs Suricata',  asm['latency_mean_us'], sur['latency_mean_us']),
    ('CPU:        ASM vs Suricata',  asm['cpu_util_pct'],    sur['cpu_util_pct']),
]

print(f'Welch Two-Sample t-Test Results (N=30, independent unpaired)')
print(f'{'Comparison':<38} {'t-stat':>8} {'p-value':>12} {"Cohen's d":>10} {'Sig':>6}')
print('-'*80)
for name, a, b in comparisons:
    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)
    d = cohens_d(a.values, b.values)
    sig = '***' if p_val < 1e-15 else ('**' if p_val < 0.001 else '*')
    p_str = f'{p_val:.2e}' if p_val > 0 else '<1e-300'
    print(f'{name:<38} {t_stat:>8.2f} {p_str:>12} {abs(d):>10.2f} {sig:>6}')

## 4. Wilson 95% Confidence Intervals (Proportions)

In [ ]:
from statsmodels.stats.proportion import proportion_confint

# LLM evaluation Wilson CIs
print('Wilson 95% CI for Edge-LLM Evaluation Proportions:')
cases = [
    ('Recall (29/29 malicious correct)', 29, 29),
    ('Precision (29/31 predicted malicious correct)', 29, 31),
    ('Overall accuracy (48/50)', 48, 50),
]
for name, k, n in cases:
    lo, hi = proportion_confint(k, n, alpha=0.05, method='wilson')
    print(f'  {name}')
    print(f'    Point: {k/n:.4f}  95% CI: [{lo:.4f}, {hi:.4f}]')
    print()

## 5. Summary Statistics Table (Paper-Ready)

In [ ]:
print('\nTable III: Comparative Benchmark (mean ± std, N=30)\n')
fw_labels = {'iptables':'Netfilter (iptables 1.8.7)',
             'Suricata Inline':'Suricata 7.0.2',
             'DPDK':'Intel DPDK 22.11',
             'ASM-Shadhin-AI (eBPF/XDP)':'ASM-Shadhin-AI'}
cols = ['throughput_mpps','latency_mean_us','jitter_us','cpu_util_pct']
col_labels = ['Throughput (Mpps)', 'Latency (µs)', 'Jitter (µs)', 'CPU (%)']
for fw in frameworks:
    sub = df[df['framework']==fw]
    label = fw_labels[fw]
    vals_str = ' | '.join([f'{np.mean(sub[c].values):.3f}±{np.std(sub[c].values,ddof=1):.3f}' for c in cols])
    print(f'{label:<30}: {vals_str}')
print(f'\nColumns: {" | ".join(col_labels)}')
print('\n--- DONE: All statistics computed and verified ---')